# CKP02 — DocMind RAG

**Disciplina:** Prompt Engineering and Artificial Intelligence · FIAP · 2º semestre

**Domínio:** recarga de veículos elétricos (carregadores GoodWe HCA G2 e norma de conexão da distribuidora Energisa)

## Integrantes

| Nome | RM |
|------|----|
| Aline Medri Marcolino | RM:569349 |
| Luis Fernando de Azevedo | RM:574167 |


## O que este notebook faz

Pipeline RAG completo: `load → split → embed → store → retrieve → generate`, com:

- **Embeddings:** `nomic-embed-text` (OllamaEmbeddings)
- **Modelo de chat:** `gemma4:cloud` via Ollama Cloud, `temperature=0`
- **Banco vetorial:** ChromaDB
- **Splitter:** `RecursiveCharacterTextSplitter` com overlap de 12% do `chunk_size`
- **Avaliação:** RAGAS (faithfulness e answer_relevancy) para 3 tamanhos de chunk e 1 variante com reranking
- **Diferenciais:** metadata filtering (`where`), reranking (cross-encoder) e interface Gradio

## Antes de executar

1. Em **Secrets** (ícone de chave, à esquerda), crie `OLLAMA_API_KEY` e ative o acesso do notebook.
2. Suba os 6 PDFs da base para a pasta `/content/data` (ícone de pasta, à esquerda).
3. Rode **Ambiente de execução → Executar tudo**.

## 1. Instalação

In [11]:
!pip -q install "langchain-community>=0.4,<0.4.2" langchain-chroma langchain-ollama langchain-text-splitters chromadb pypdf sentence-transformers "ragas>=0.4,<0.5" pandas gradio

## 2. Configuração dos modelos (Ollama Cloud)

A chave `OLLAMA_API_KEY` é lida dos **Secrets** do Colab, nunca fica escrita no código.

In [12]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

!ollama pull nomic-embed-text

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [13]:
import os
from pathlib import Path

import pandas as pd
from langchain_ollama import ChatOllama, OllamaEmbeddings

try:
    from google.colab import userdata
    OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY").strip()
except Exception:
    OLLAMA_API_KEY = (os.environ.get("OLLAMA_API_KEY") or "").strip()

assert OLLAMA_API_KEY, "Crie o Secret OLLAMA_API_KEY (ícone de chave) e ative o acesso do notebook."

OLLAMA_URL = "https://ollama.com"          # endpoint do Ollama Cloud (só para o chat)
CABECALHO = {"headers": {"Authorization": f"Bearer {OLLAMA_API_KEY}"}}

# embeddings: Ollama instalado no próprio Colab (célula anterior)
embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
# chat: gemma4:cloud pelo Ollama Cloud, com a chave
llm = ChatOllama(model="gemma4:cloud", temperature=0, base_url=OLLAMA_URL, client_kwargs=CABECALHO)

print("Embedding de teste:", len(embeddings.embed_query("teste")), "dimensões")
print("LLM de teste:", llm.invoke("Responda apenas: ok").content)

Embedding de teste: 768 dimensões
LLM de teste: ok


## 3. Base de conhecimento (documentos reais)

Os documentos ficam em `/content/data`. Cada arquivo recebe **metadata** (`tipo`, `emissor`, `categoria`), usada depois no filtro `where`.

| Arquivo | Documento | Emissor | Link de origem |
|---------|-----------|---------|----------------|
| `goodwe_hca_g2_datasheet.pdf` | Ficha técnica da linha HCA G2 (carregador CA 7/11/22 kW) | GoodWe | [colar link] |
| `goodwe_hca_g2_manual.pdf` | Manual do usuário da linha HCA G2 | GoodWe | [colar link] |
| `goodwe_hca_g2_guia_rapido.pdf` | Guia rápido de instalação | GoodWe | [colar link] |
| `goodwe_sems_plus_app.pdf` | Guia do aplicativo SEMS+ | GoodWe | [colar link] |
| `goodwe_compatibilidade.pdf` | Lista de compatibilidade entre carregadores e inversores (v1, 15/08/2026) | GoodWe | [colar link] |
| `energisa_ndu042.pdf` | NDU-042: Fornecimento de energia para estações de recarga de veículo elétrico (v1.0, jun/2022) | Grupo Energisa | [colar link] |

In [10]:
from langchain_community.document_loaders import PyPDFLoader

DATA_DIR = "/content/data"

# metadata de cada documento (base do metadata filtering)
FONTES = {
    "goodwe_hca_g2_datasheet.pdf":   {"tipo": "datasheet",       "emissor": "GoodWe",  "categoria": "equipamento"},
    "goodwe_hca_g2_manual.pdf":      {"tipo": "manual",          "emissor": "GoodWe",  "categoria": "equipamento"},
    "goodwe_hca_g2_guia_rapido.pdf": {"tipo": "guia_instalacao", "emissor": "GoodWe",  "categoria": "equipamento"},
    "goodwe_sems_plus_app.pdf":      {"tipo": "guia_app",        "emissor": "GoodWe",  "categoria": "aplicativo"},
    "goodwe_compatibilidade.pdf":    {"tipo": "compatibilidade", "emissor": "GoodWe",  "categoria": "equipamento"},
    "energisa_ndu042.pdf":           {"tipo": "norma",           "emissor": "Energisa", "categoria": "regulacao"},
}


def carregar_documentos(pasta=DATA_DIR):
    """Carrega os PDFs da base e anexa a metadata de cada um."""
    docs = []
    for nome, meta in FONTES.items():
        caminho = Path(pasta) / nome
        if not caminho.exists():
            print(f"AVISO: {nome} não encontrado em {pasta}")
            continue
        paginas = PyPDFLoader(str(caminho)).load()
        for p in paginas:
            p.metadata.update({"fonte": nome, **meta})
        docs.extend(paginas)
        print(f"{nome}: {len(paginas)} páginas")
    return docs


docs = carregar_documentos()
assert docs, "Nenhum documento carregado: suba os PDFs para /content/data."
print("Total de páginas:", len(docs))

goodwe_hca_g2_datasheet.pdf: 2 páginas
goodwe_hca_g2_manual.pdf: 95 páginas
goodwe_hca_g2_guia_rapido.pdf: 2 páginas
goodwe_sems_plus_app.pdf: 51 páginas
goodwe_compatibilidade.pdf: 2 páginas
energisa_ndu042.pdf: 22 páginas
Total de páginas: 174


## 4. Split (chunking)

`RecursiveCharacterTextSplitter` com `separators=["\n\n", "\n", ". ", " ", ""]` e `chunk_overlap` de 12% do `chunk_size`.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def dividir(documentos, chunk_size):
    """Divide os documentos em chunks, com overlap de 12% do chunk_size."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size * 0.12),
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documentos)
    return [c for c in chunks if c.page_content.strip()]


for tam in (256, 512, 1024):
    print(f"chunk_size={tam}: {len(dividir(docs, tam))} chunks")

chunk_size=256: 634 chunks
chunk_size=512: 348 chunks
chunk_size=1024: 224 chunks


## 5. Embed + store (ChromaDB)

In [15]:
from langchain_chroma import Chroma


def criar_colecao(chunks, nome):
    """Gera os embeddings (nomic-embed-text) e grava os chunks numa coleção do Chroma."""
    try:  # permite reexecutar a célula sem erro de coleção duplicada
        Chroma(collection_name=nome, embedding_function=embeddings).delete_collection()
    except Exception:
        pass
    return Chroma.from_documents(chunks, embeddings, collection_name=nome)


chunks_512 = dividir(docs, 512)
VS_PADRAO = criar_colecao(chunks_512, "carregadores_ev_512")
print("Coleção criada com", len(chunks_512), "chunks")
print("Busca semântica de teste:", VS_PADRAO.similarity_search("potência do GW11K-HCA-20", k=1)[0].page_content[:200])

Coleção criada com 348 chunks
Busca semântica de teste: GW7K-HCA-20 opcional
GW11K-HCA-20 e GW22K-HCA-20 opcionais
33


## 6. Retrieve (com metadata filtering e reranking)

`buscar(consulta)` é a função reutilizada como `@tool` no CKP03.

- `tipo="norma"` aplica o filtro `where` do Chroma pela metadata.
- `rerank=True` recupera mais candidatos e reordena com o cross-encoder `ms-marco-MiniLM-L-6-v2`.

In [16]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def buscar(consulta, vs=None, k=4, rerank=False, tipo=None):
    """Retorna os k trechos mais relevantes para a consulta.

    tipo: filtra pela metadata (ex.: "norma", "datasheet", "manual").
    rerank: reordena os candidatos com o cross-encoder antes de devolver.
    """
    if vs is None:
        vs = VS_PADRAO
    filtro = {"tipo": tipo} if tipo else None
    n_candidatos = k * 4 if rerank else k
    candidatos = vs.similarity_search(consulta, k=n_candidatos, filter=filtro)
    if rerank and candidatos:
        notas = reranker.predict([(consulta, c.page_content) for c in candidatos])
        ordenados = sorted(zip(candidatos, notas), key=lambda par: -par[1])
        candidatos = [c for c, _ in ordenados][:k]
    return candidatos


# metadata filtering: a mesma pergunta, só nos documentos do tipo "norma"
for c in buscar("grau de proteção em ambiente externo", tipo="norma", k=2):
    print(c.metadata["fonte"], "p.", c.metadata.get("page", 0) + 1, "|", c.page_content[:120].replace("\n", " "))

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

energisa_ndu042.pdf p. 18 | individualmente por um dispositivo de proteção à corrente d iferencial-
energisa_ndu042.pdf p. 19 | proteções, propõe -se que ao adquirir a estação de recarga, elas sejam  confirmadas junto ao fornecedor/fabricante;


## 7. Generate (resposta fundamentada, citando a fonte)

In [17]:
PROMPT = """Você é um assistente técnico sobre carregadores de veículos elétricos.
Responda APENAS com base nos trechos abaixo. Se a informação não estiver nos trechos, diga que não encontrou nos documentos.
Cite a fonte de cada afirmação no formato [arquivo, p. N].

Trechos:
{contexto}

Pergunta: {pergunta}
Resposta:"""


def formatar_contexto(trechos):
    """Monta o contexto numerando cada trecho com arquivo e página de origem."""
    return "\n\n".join(
        f"[{t.metadata['fonte']}, p. {t.metadata.get('page', 0) + 1}]\n{t.page_content}" for t in trechos
    )


def gerar(pergunta, trechos):
    """Gera a resposta do LLM (temperature=0) usando só os trechos recuperados."""
    return llm.invoke(PROMPT.format(contexto=formatar_contexto(trechos), pergunta=pergunta)).content


def responder(pergunta, **kwargs):
    """Pipeline completo de consulta: retrieve + generate."""
    trechos = buscar(pergunta, **kwargs)
    return gerar(pergunta, trechos), trechos


pergunta = "Os veículos elétricos podem injetar energia de volta na rede de distribuição?"
resposta, trechos = responder(pergunta, rerank=True)
print(resposta)
print("\nFontes:", sorted({(t.metadata["fonte"], t.metadata.get("page", 0) + 1) for t in trechos}))

Não, é vedada a injeção de energia elétrica na rede de distribuição a partir dos veículos elétricos [energisa_ndu042.pdf, p. 11].

Fontes: [('energisa_ndu042.pdf', 11), ('energisa_ndu042.pdf', 16), ('energisa_ndu042.pdf', 20), ('goodwe_hca_g2_manual.pdf', 79)]


## 8. Avaliação com RAGAS

8 perguntas sobre os documentos. Para cada configuração de chunking, o pipeline responde as perguntas e o RAGAS mede:

- **faithfulness:** a resposta está sustentada pelos trechos recuperados?
- **answer_relevancy:** a resposta atende à pergunta?

In [18]:
PERGUNTAS = [
    "Quais são os modos de carregamento do carregador GoodWe HCA e como funciona o modo PV + Bateria?",
    "Quais modelos fazem parte da linha HCA G2 e qual a potência de cada um?",
    "Como iniciar e encerrar uma recarga pelo aplicativo da GoodWe?",
    "A instalação de uma estação de recarga precisa ser comunicada à distribuidora? Em quais casos?",
    "É permitido cobrar pela recarga de veículos que não são do titular da unidade consumidora?",
    "Os veículos elétricos podem injetar energia de volta na rede de distribuição?",
    "Quais graus de proteção a norma da Energisa exige para estações de recarga instaladas em ambiente externo?",
    "O que o consumidor precisa fazer antes de instalar um carregador GoodWe em casa, segundo a regulação e a norma da distribuidora?",
]

# (nome, chunk_size, usar_rerank)
VARIANTES = [
    ("chunk256", 256, False),
    ("chunk512", 512, False),
    ("chunk1024", 1024, False),
    ("chunk512_rerank", 512, True),
]

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.run_config import RunConfig

Path("resultados").mkdir(exist_ok=True)
juiz_llm = LangchainLLMWrapper(llm)
juiz_emb = LangchainEmbeddingsWrapper(embeddings)

colecoes, resumo, tabelas = {}, [], {}

for nome, tam, rerank in VARIANTES:
    if tam not in colecoes:
        chunks = dividir(docs, tam)
        colecoes[tam] = (criar_colecao(chunks, f"carregadores_ev_{tam}"), len(chunks))
    vs, n_chunks = colecoes[tam]

    print(f"\n== Gerando respostas: {nome} ==")
    amostras = []
    for p in PERGUNTAS:
        trechos = buscar(p, vs=vs, rerank=rerank)
        amostras.append({
            "user_input": p,
            "response": gerar(p, trechos),
            "retrieved_contexts": [t.page_content for t in trechos],
        })

    print(f"== Avaliando com RAGAS: {nome} ==")
    res = evaluate(
        dataset=EvaluationDataset.from_list(amostras),
        metrics=[Faithfulness(), ResponseRelevancy()],
        llm=juiz_llm,
        embeddings=juiz_emb,
        run_config=RunConfig(timeout=300, max_workers=2),
    )
    df = res.to_pandas()
    df.to_csv(f"resultados/ragas_{nome}.csv", index=False)
    tabelas[nome] = df

    resumo.append({
        "variante": nome,
        "chunk_size": tam,
        "rerank": rerank,
        "n_chunks": n_chunks,
        "faithfulness_medio": df["faithfulness"].mean(),
        "answer_relevancy_medio": df["answer_relevancy"].mean(),
        "perguntas_com_NaN": int(df[["faithfulness", "answer_relevancy"]].isna().any(axis=1).sum()),
    })

tabela_resumo = pd.DataFrame(resumo)
tabela_resumo.to_csv("resultados/resumo.csv", index=False)
print("\n===== RESUMO =====")
tabela_resumo

/tmp/ipykernel_1329/1115938766.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
/tmp/ipykernel_1329/1115938766.py:4: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy
/tmp/ipykernel_1329/1115938766.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  juiz_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_1329/1115938766.py:9: Deprecat


== Gerando respostas: chunk256 ==
== Avaliando com RAGAS: chunk256 ==


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

### Tabela por pergunta (faithfulness e answer_relevancy) para cada configuração

In [ ]:
for nome, df in tabelas.items():
    print(f"\n--- {nome} ---")
    print(df[["user_input", "faithfulness", "answer_relevancy"]].to_string())

In [ ]:
melhor = tabela_resumo.sort_values("faithfulness_medio", ascending=False).iloc[0]
print(f"Maior faithfulness média: {melhor['variante']} ({melhor['faithfulness_medio']:.3f})")
print("Todas as variantes com faithfulness >= 0,7:", bool((tabela_resumo["faithfulness_medio"] >= 0.7).all()))

## 9. Conclusão

**Escolha final:** [preencher depois de rodar, com base na tabela acima]

[Justifique em 2 a 3 linhas: qual variante teve a maior faithfulness, por que o tamanho do chunk ou o reranking ajudou e o que aconteceu nas perguntas com answer_relevancy igual a 0,0.]

**Limitações:** a amostra tem 8 perguntas, então uma única pergunta muda a média de faithfulness em 0,125 e as diferenças pequenas entre as variantes não são conclusivas.

## 9. Conclusão

**Escolha final: chunk_size = 512.**

Com chunk de 512 a faithfulness média foi 1,000, contra 0,813 com 256 e 0,844 com 1024. Em 256 e 1024, a pergunta sobre a injeção de energia na rede teve faithfulness 0,0, enquanto em 512 foi 1,0. Chunks de 256 fragmentam as regras da norma (633 chunks) e chunks de 1024 misturam assuntos no mesmo trecho, o que reduz a precisão da recuperação. Todas as variantes ficaram acima da meta de faithfulness (≥ 0,7).

O reranking com cross-encoder (`chunk512_rerank`) obteve faithfulness equivalente (0,969). A diferença vem de uma única pergunta e não indica que um seja melhor que o outro nesta amostra, então o reranking ficou como diferencial, sem ganho adicional medido.

**Limitações:**
- A amostra tem 8 perguntas, então uma única pergunta muda a média de faithfulness em 0,125 e diferenças pequenas entre as variantes não são conclusivas.
- O avaliador do RAGAS é um LLM, e há variação entre execuções.
- As perguntas sobre o uso do aplicativo e sobre os graus de proteção da norma tiveram answer_relevancy 0,0 em todas as variantes, com faithfulness 1,0. Isso significa que as respostas não contradizem os trechos recuperados, mas o avaliador não as considerou como respondendo à pergunta. Esse padrão é compatível com respostas incompletas por falha na recuperação do trecho certo, e não com alucinação.

## 10. Interface Gradio (diferencial)

Chat com o RAG integrado, usando a variante escolhida (`chunk512` com reranking) e mostrando a fonte na resposta.

In [ ]:
import gradio as gr

VS_FINAL = colecoes[512][0]


def conversar(mensagem, historico):
    trechos = buscar(mensagem, vs=VS_FINAL, rerank=True)
    resposta = gerar(mensagem, trechos)
    fontes = sorted({f"{t.metadata['fonte']} (p. {t.metadata.get('page', 0) + 1})" for t in trechos})
    return resposta + "\n\n**Fontes:** " + "; ".join(fontes)


demo = gr.ChatInterface(conversar, title="DocMind RAG — Carregadores GoodWe HCA G2")
demo.launch(prevent_thread_lock=True)

## 11. Como adicionar novos documentos

1. Suba o arquivo (PDF) para `/content/data`.
2. Registre o arquivo e sua metadata no dicionário `FONTES` (seção 3) e na tabela de fontes.
3. Rode o notebook de novo com **Executar tudo**: o split, os embeddings e a avaliação são refeitos com o novo documento.